<h3 style="color:#6FA8DC; font-weight:bold">Fetching Data from APIs using Python, Requests and Pandas</h3>

Learn how to fetch data from an API, convert JSON into a Pandas DataFrame, use parameters, handle errors, clean data, work with nested JSON and save results.

We will use the **GFG BVCOE Events API**.

<h5 style="color:#78B89A; font-weight:bold;">1. What is an API? → Simple meaning</h5>

An API allows two applications to communicate.

```text
Jupyter Notebook → API Request → Backend → JSON Response → Pandas DataFrame
```

<h5 style="color:#78B89A; font-weight:bold;">2. Import libraries</h5>

- `requests` fetches data from APIs.
- `pandas` cleans and analyzes data.

In [ ]:
import requests
import pandas as pd

<h5 style="color:#78B89A; font-weight:bold;">3. Store the API URL</h5>

In [ ]:
url = "https://gfgxbvcoe.onrender.com/api/v1/events"
url

<h5 style="color:#78B89A; font-weight:bold;">4. Send a GET request</h5>

`GET` is used to fetch data.

In [ ]:
res = requests.get(url)
res

<h5 style="color:#78B89A; font-weight:bold;">5. Check status code</h5>

| Code | Meaning |
|---|---|
| 200 | Success |
| 201 | Created |
| 400 | Bad request |
| 401 | Unauthorized |
| 403 | Forbidden |
| 404 | Not found |
| 500 | Server error |
| 503 | Service unavailable |

In [ ]:
res.status_code

In [ ]:
if res.status_code == 200:
    print("Data fetched successfully")
else:
    print("Something went wrong")

<h5 style="color:#78B89A; font-weight:bold;">6. Convert response into JSON</h5>

`res.json()` converts the response into Python dictionaries/lists.

In [ ]:
data = res.json()
type(data)

In [ ]:
data.keys()

<h5 style="color:#78B89A; font-weight:bold;">7. Extract event data</h5>

In [ ]:
events = data["data"]
type(events)

In [ ]:
events[0]

In [ ]:
events[0].keys()

<h5 style="color:#78B89A; font-weight:bold;">8. Convert JSON into DataFrame</h5>

In [ ]:
df = pd.DataFrame(events)
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

<h5 style="color:#78B89A; font-weight:bold;">9. Useful response properties</h5>

- `res.text` → response as text
- `res.content` → response as bytes
- `res.headers` → response headers
- `res.url` → final URL
- `res.ok` → success status
- `res.reason` → status explanation

In [ ]:
res.text[:500]

In [ ]:
res.content[:100]

In [ ]:
res.headers

In [ ]:
res.url

In [ ]:
res.ok

In [ ]:
res.reason

<h5 style="color:#78B89A; font-weight:bold;">10. Raise errors automatically</h5>

In [ ]:
res.raise_for_status()
print("Request was successful")

<h5 style="color:#78B89A; font-weight:bold;">11. Error handling</h5>

In [ ]:
try:
    res = requests.get(url, timeout=15)
    res.raise_for_status()

    api_data = res.json()
    df = pd.DataFrame(api_data["data"])

    print("Data fetched successfully")
    display(df.head())

except requests.exceptions.Timeout:
    print("The API took too long to respond")
except requests.exceptions.RequestException as e:
    print("Request error:", e)
except ValueError:
    print("Response was not valid JSON")
except KeyError:
    print("The expected 'data' key was not found")

<h5 style="color:#78B89A; font-weight:bold;">12. Query parameters using `params`</h5>

Query parameters customize a request. They work only if the backend supports them.

In [ ]:
params = {
    "limit": 10,
    "page": 1
}

res_with_params = requests.get(
    url,
    params=params,
    timeout=15
)

print(res_with_params.url)

<h5 style="color:#78B89A; font-weight:bold;">13. Request headers</h5>

In [ ]:
headers = {
    "Accept": "application/json"
}

res = requests.get(
    url,
    headers=headers,
    timeout=15
)

res.status_code

For authenticated APIs:

```python
headers = {
    "Authorization": "Bearer YOUR_TOKEN"
}
```

Never expose real API tokens.

<h5 style="color:#78B89A; font-weight:bold;">14. Timeout</h5>

In [ ]:
res = requests.get(url, timeout=10)
res.status_code

<h5 style="color:#78B89A; font-weight:bold;">15. Select useful columns</h5>

In [ ]:
selected_columns = ["title", "date", "time", "location", "category"]

events_summary = df[
    [col for col in selected_columns if col in df.columns]
]

events_summary.head()

<h5 style="color:#78B89A; font-weight:bold;">16. Missing values</h5>

In [ ]:
df.isnull().sum()

In [ ]:
df[df.isnull().any(axis=1)].head()

<h5 style="color:#78B89A; font-weight:bold;">17. Convert dates</h5>

In [ ]:
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

df[["date"]].head()

In [ ]:
if "date" in df.columns:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month_name()
    df["weekday"] = df["date"].dt.day_name()

df.head()

<h5 style="color:#78B89A; font-weight:bold;">18. Filter API data</h5>

In [ ]:
if "category" in df.columns:
    display(df[df["category"] == "Workshop"])

In [ ]:
if "location" in df.columns:
    display(df[df["location"] == "B-401, BVCOE"])

In [ ]:
if {"category", "location"}.issubset(df.columns):
    display(df[
        (df["category"] == "Workshop") &
        (df["location"] == "B-401, BVCOE")
    ])

<h5 style="color:#78B89A; font-weight:bold;">19. Sort and count</h5>

In [ ]:
if "date" in df.columns:
    display(df.sort_values("date", ascending=False).head())

In [ ]:
if "category" in df.columns:
    display(df["category"].value_counts())

In [ ]:
if "location" in df.columns:
    display(df["location"].value_counts())

<h5 style="color:#78B89A; font-weight:bold;">20. Flatten nested JSON</h5>

Use `pd.json_normalize()` for nested dictionaries.

In [ ]:
flat_df = pd.json_normalize(events)
flat_df.head()

In [ ]:
sample_data = [{
    "title": "Python Workshop",
    "organizer": {
        "name": "GFG BVCOE",
        "email": "gfg@example.com"
    }
}]

pd.json_normalize(sample_data)

<h5 style="color:#78B89A; font-weight:bold;">21. Expand nested lists</h5>

If `speakers` contains a list of dictionaries, use `record_path`.

In [ ]:
if "speakers" in df.columns:
    speakers_df = pd.json_normalize(
        events,
        record_path="speakers",
        meta=["title", "date"],
        errors="ignore"
    )
    display(speakers_df.head())

<h5 style="color:#78B89A; font-weight:bold;">22. Pagination</h5>

This general example works only if the API supports `page` and `limit`.

In [ ]:
all_events = []

for page in range(1, 4):
    params = {"page": page, "limit": 10}

    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()

    all_events.extend(response.json()["data"])

all_events_df = pd.DataFrame(all_events)
all_events_df.head()

<h5 style="color:#78B89A; font-weight:bold;">23. Save as CSV</h5>

In [ ]:
df.to_csv("events_from_api.csv", index=False)

<h5 style="color:#78B89A; font-weight:bold;">24. Save as JSON</h5>

In [ ]:
df.to_json("events_from_api.json", orient="records", indent=4)

<h5 style="color:#78B89A; font-weight:bold;">25. Other HTTP methods</h5>

| Method | Purpose |
|---|---|
| GET | Fetch data |
| POST | Create data |
| PUT | Replace data |
| PATCH | Partially update data |
| DELETE | Delete data |

In [ ]:
# POST example
new_event = {
    "title": "Python Workshop",
    "category": "Workshop"
}

# requests.post(url, json=new_event)

In [ ]:
# PUT example
# requests.put(
#     "https://example.com/events/event_id",
#     json={"title": "Updated Workshop"}
# )

In [ ]:
# PATCH example
# requests.patch(
#     "https://example.com/events/event_id",
#     json={"title": "New Title"}
# )

In [ ]:
# DELETE example
# requests.delete("https://example.com/events/event_id")

**Important:** Use POST, PUT, PATCH and DELETE only when you have permission to modify backend data.

<h3 style="color:#6FA8DC; font-weight:bold">Quick Revision</h3>

| Function | Use |
|---|---|
| `requests.get()` | Fetch data |
| `res.json()` | Convert response to JSON |
| `res.status_code` | Check status |
| `res.raise_for_status()` | Raise HTTP errors |
| `res.text` | Read text |
| `res.headers` | Read headers |
| `params` | Query parameters |
| `headers` | Request headers |
| `timeout` | Limit waiting |
| `pd.DataFrame()` | Convert JSON to DataFrame |
| `pd.json_normalize()` | Flatten nested JSON |
| `df.to_csv()` | Save CSV |
| `df.to_json()` | Save JSON |

### Most important workflow

```text
requests.get()
      ↓
res.json()
      ↓
Extract "data"
      ↓
pd.DataFrame()
      ↓
Clean and analyze
```